# Inspecting model outputs and scores

This notebook walks through the sample predictions shipped in
`results/samples/` (five examples from three key experiments) and shows how
the evaluation in `evaluation/compute_metrics.py` parses and scores them.

In [ ]:
import json
import re
from pathlib import Path

REPO = Path("..").resolve()

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]

samples = load_jsonl(REPO / "results/samples/hateful/grpo_think_lenreg.jsonl")
len(samples), list(samples[0].keys())

## Anatomy of a prediction

The trained model emits a private reasoning trace inside `<think>` tags,
then the label and a reference-style explanation.

In [ ]:
response = samples[0]["response"]
think = re.search(r"<think>(.*?)</think>", response, re.DOTALL)
print("--- think (truncated) ---")
print(think.group(1).strip()[:600], "...")
print()
print("--- answer ---")
print(response[think.end():].strip())

In [ ]:
LABEL_RE = re.compile(r"Label:\s*([^\n]+)", re.IGNORECASE)

for i, s in enumerate(samples):
    pred = LABEL_RE.search(s["response"]).group(1).strip()
    gold = LABEL_RE.search(s["labels"]).group(1).strip()
    print(f"{i}: pred={pred:<12} gold={gold:<12} {'ok' if pred == gold else 'WRONG'}")

## Scores

`scores/` holds the metrics computed on the *full* test sets for the same
experiments (these are the numbers reported in the paper).

In [ ]:
import pandas as pd

rows = []
for metrics_file in sorted(REPO.glob("scores/*/*/metrics.json")):
    m = json.load(open(metrics_file))
    rows.append({
        "experiment": f"{metrics_file.parent.parent.name}/{metrics_file.parent.name}",
        "accuracy": m["accuracy"],
        "f1_macro": m["f1_macro"],
        "f1_weighted": m["f1_weighted"],
        "bertscore_f1": m.get("bertscore_f1"),
        "meteor": m.get("meteor"),
    })
pd.DataFrame(rows).round(3)

The GRPO + $R_{think}$ run reaches 82.1 accuracy / 0.806 macro-F1 on Hateful
Memes and the self-supervised run 0.612 macro-F1 on ArMeme - matching
Tables 9-11 of the paper.

To reproduce these metrics from a prediction file:

```bash
python evaluation/compute_metrics.py \
    --data results/hateful/your_run.jsonl \
    --out_dir scores/hateful/your_run --has_explanation
```

In [ ]:
from IPython.display import Image, display

display(Image(REPO / "scores/hateful/grpo_think_lenreg/confusion_matrix.png"))